# Getting data

### Setup

In [307]:
import pandas as pd
import sqlite3

## CSV Files

###  Read a CSV into a DataFrame directly from a web URL

In [308]:
mortality_url = "https://www.murach.com/python_analysis/mortality.csv"
mortality_data = pd.read_csv(mortality_url)

In [309]:
mortality_data

,Year,Age Group,Death Rate
0,1900,1-4 Years,1983.8
1,1901,1-4 Years,1695.0
2,1902,1-4 Years,1655.7
3,1903,1-4 Years,1542.1
4,1904,1-4 Years,1591.5
...,...,...,...
471,2014,15-19 Years,45.5
472,2015,15-19 Years,48.3
473,2016,15-19 Years,51.2
474,2017,15-19 Years,51.5


###  Downloading a file to disk before loading it into a DataFrame

In [310]:
from urllib import request
polls_url = 'https://www.murach.com/python_analysis/president_polls_2016.csv'
request.urlretrieve(polls_url, filename='president_polls_2016.csv')

('president_polls_2016.csv', <http.client.HTTPMessage at 0x7ff48b0f6cd0>)

In [311]:
polls = pd.read_csv('president_polls_2016.csv')
polls.head(2)

,cycle,branch,type,matchup,forecastdate,state,startdate,enddate,pollster,grade,...,adjpoll_clinton,adjpoll_trump,adjpoll_johnson,adjpoll_mcmullin,multiversions,url,poll_id,question_id,createddate,timestamp
0,2016,President,polls-plus,Clinton vs. Trump vs. Johnson,11/8/2016,U.S.,11/3/2016,11/6/2016,ABC News/Washington Post,A+,...,45.20163,41.72430,4.626221,NaN,NaN,https://www.washingtonpost.com/news/the-fix/wp...,48630,76192,11/7/2016,11/8/2016 9:35
1,2016,President,polls-plus,Clinton vs. Trump vs. Johnson,11/8/2016,U.S.,11/1/2016,11/7/2016,Google Consumer Surveys,B,...,43.34557,41.21439,5.175792,NaN,NaN,https://datastudio.google.com/u/0/#/org//repor...,48847,76443,11/7/2016,11/8/2016 9:35


## Databases

### The Movie Database:  Table Structure

### How to run queries against a database

### List all the actors in the actors table

In [312]:
movies_con = sqlite3.connect('data/MovieDb.sqlite')
movies_cur = movies_con.cursor()
movies_cur.execute(
    'SELECT * FROM directors').fetchall()

[(1, 'Steven Spielberg'),
 (2, 'Tony Scott'),
 (3, 'Barry Levinson'),
 (4, 'Rob Reiner'),
 (5, 'Nora Ephron'),
 (6, 'Robert Zemeckis'),
 (7, 'Robert Benton'),
 (8, 'Paul Brickman'),
 (9, 'Mike Nichols'),
 (10, 'Penny Marshall')]

### Get information about a table

In [313]:
movies_cur.execute('PRAGMA table_info(actors)').fetchall()

[(0, 'id', 'INTEGER', 0, None, 0), (1, 'name', 'TEXT', 0, None, 0)]

### Import the data from a simple query into a DataFrame (Single Table)

In [314]:
movies = pd.read_sql_query(
    '''SELECT name
    FROM actors''', movies_con)
movies.head(10)

,name
0,Andy Serkis
1,Anne Bancroft
2,Anthony Edwards
3,Ariana Richards
4,BD Wong
5,Billy Crystal
6,Bob Odenkirk
7,Bob Peck
8,Bradley Whitford
9,Bruce Greenwood


### Join multiple tables and import into a DataFrame

In [315]:
moviedirector = pd.read_sql_query(
    '''SELECT release_date AS "Release Date", title AS "Title", directors.name AS "Director", budget AS "Budget"
       FROM movies
    JOIN directors ON movies.director_id = directors.id''', movies_con)
moviedirector.head(5)

,Release Date,Title,Director,Budget
0,1986,Top Gun,Tony Scott,15000000.0
1,1989,When Harry Met Sally,Rob Reiner,16000000.0
2,1988,Rain Man,Barry Levinson,25000000.0
3,1988,You've Got Mail,Nora Ephron,65000000.0
4,1994,Forest Gump,Robert Zemeckis,55000000.0


# Cleaning Data

### Types of problems with data
- Duplicate rows
- Rows that are not needed
- Columns that are not needed
- Rename the columns so easier to understand
- Missing values
- Wrong data types like numbers that are imported as strings
- Find and fix outliers

### Setup

##### Note:  We will overwrite the previous version of the mortality data

In [316]:
mortality_data = pd.read_csv('data/mortality.csv')
mortality_data.columns = mortality_data.columns.str.replace(" ", "")
mortality_data.AgeGroup = mortality_data.AgeGroup.replace( {'1-4 Years':'01-04 Years','5-9 Years':'05-09 Years'})
mortality_data['MeanCentered'] = mortality_data.DeathRate - mortality_data.DeathRate.mean()

In [317]:
mortality_data.head()

,Year,AgeGroup,DeathRate,MeanCentered
0,1900,01-04 Years,1983.8,1790.87584
1,1901,01-04 Years,1695.0,1502.07584
2,1902,01-04 Years,1655.7,1462.77584
3,1903,01-04 Years,1542.1,1349.17584
4,1904,01-04 Years,1591.5,1398.57584


In [318]:
mortality_data.tail()

,Year,AgeGroup,DeathRate,MeanCentered
471,2014,15-19 Years,45.5,-147.42416
472,2015,15-19 Years,48.3,-144.62416
473,2016,15-19 Years,51.2,-141.72416
474,2017,15-19 Years,51.5,-141.42416
475,2018,15-19 Years,49.2,-143.72416


### Load the mortality data to be cleaned 

### Summary of problems
###### Note: The header is row 1
- There is a column (Current date) that is not needed.  This might be analogous to a metadata time stamp for when the data was collected.
- Data type of the year column
- One of the "Age Group" values has a typo. 
- Row 11:  Data missing
- Row 16: Data missing
- Row 21: Data missing
- Row 30: Data missing
- Row 38 Data missing
- Row 51: Data missing
- Row 67: Data missing
- Row 68 - 69 (year 1966):  duplicate row
- Row 83: Data missing
- Row 87 - Row 89:  duplicate rows
- Row 148: Data missing
- Row 212: invalid numeric data
- Row 250: invalid string data in the DeathRate cell
- Row 317: date and data are missing (drop)


##### NOTE:  Dropping the "Mean Centered" column because this fails with invalid data types.
- This data would need to be cleaned before adding this column

In [319]:
mortality_to_clean = pd.read_csv('data/mymortality-messy.csv')
mortality_to_clean.columns = mortality_to_clean.columns.str.replace(" ", "")
mortality_to_clean.AgeGroup = mortality_to_clean.AgeGroup.replace( {'1-4 Years':'01-04 Years','5-9 Years':'05-09 Years'})
# mortality_to_clean['MeanCentered'] = mortality_to_clean.DeathRate - mortality_to_clean.DeathRate.mean()

In [320]:
mortality_to_clean.head(5)

,Year,AgeGroup,DeathRate,CurrentDate
0,1900.0,1-4 years,1983.8,6/20/2025
1,1901.0,01-04 Years,1695,6/21/2025
2,1902.0,01-04 Years,1655.7,6/22/2025
3,1903.0,01-04 Years,1542.1,6/23/2025
4,1904.0,01-04 Years,1591.5,6/24/2025


### (Optional) Compare with the other data sets too 
#####  These take a while to load, so uncomment when you are ready, but have coffee nearby. 

##### Optional data sets (uncomment below to use)
- Polls
- Jobs

 

In [321]:
# This takes a really long time to run
# polls = pd.read_csv('data/president_polls_2016.csv')

In [322]:
# This takes a really long time to run
# jobs = pd.read_excel('data/all_data_M_2018.xlsx')

In [323]:
#polls.head(3)

In [324]:
# jobs.head(3)

### Run info() on all of the data
##### Compare the results
- mortality (original)
- mortality (needs to be cleaned) 

##### Mortality (orignal)

In [325]:
mortality_data.info(verbose=True, memory_usage='deep', show_counts=True)                

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 476 entries, 0 to 475
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Year          476 non-null    int64  
 1   AgeGroup      476 non-null    object 
 2   DeathRate     476 non-null    float64
 3   MeanCentered  476 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 42.9 KB


In [326]:
mortality_data.nunique()

Year            119
AgeGroup          4
DeathRate       430
MeanCentered    430
dtype: int64

- Returns each unique value in a column

In [327]:
mortality_data.Year.value_counts().head(10)

Year
1900    4
1975    4
1987    4
1986    4
1985    4
1984    4
1983    4
1982    4
1981    4
1980    4
Name: count, dtype: int64

##### Mortality (To Clean)

In [328]:
mortality_to_clean.info(verbose=True, memory_usage='deep', show_counts=True)          

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483 entries, 0 to 482
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Year         482 non-null    float64
 1   AgeGroup     483 non-null    object 
 2   DeathRate    473 non-null    object 
 3   CurrentDate  483 non-null    object 
dtypes: float64(1), object(3)
memory usage: 95.6 KB


- We can see the extra columns
- Notice "AgeGroup."  One of the data points in "Age Group" has a typo, "Years" vs. "years."

In [329]:
mortality_to_clean.nunique()

Year           119
AgeGroup         5
DeathRate      420
CurrentDate    443
dtype: int64

- Each year should have a count of 4 (representing the 4 valid age groups)

In [330]:
mortality_to_clean.Year.value_counts().head(10)

Year
1984.0    10
1966.0     5
1900.0     4
1975.0     4
1987.0     4
1986.0     4
1985.0     4
1983.0     4
1982.0     4
1981.0     4
Name: count, dtype: int64

### Drop Duplicated Rows
- This is a little tricky because the year 1966 for Age Group 1 to 4 years is also a duplicated row, but the extra column (which has different dates (timestamps) for these two years, makes it appear like it is how a duplicate.
- We want to see what this looks like after we drop the CurrentDate column.

In [331]:
mortality_to_clean[mortality_to_clean.duplicated(keep=False)]

,Year,AgeGroup,DeathRate,CurrentDate
444,1984.0,15-19 Years,80.4,9/30/2023
445,1984.0,15-19 Years,80.4,9/30/2023
446,1984.0,15-19 Years,80.4,9/30/2023
447,1984.0,15-19 Years,80.4,9/30/2023
448,1984.0,15-19 Years,80.4,9/30/2023


- Use the nunique() method to find unwanted columns and then drop them.

In [332]:
mortality_to_clean.nunique()

Year           119
AgeGroup         5
DeathRate      420
CurrentDate    443
dtype: int64

In [333]:
mortality_to_clean = mortality_to_clean.drop(columns=['CurrentDate'])
mortality_to_clean.nunique()

Year         119
AgeGroup       5
DeathRate    420
dtype: int64

- Try running the duplicate method again
- Now we see the duplicated rows in year 1966

In [334]:
mortality_to_clean[mortality_to_clean.duplicated(keep=False)]

,Year,AgeGroup,DeathRate
66,1966.0,01-04 Years,96.4
67,1966.0,01-04 Years,96.4
85,1984.0,01-04 Years,52.2
86,1984.0,01-04 Years,52.2
87,1984.0,01-04 Years,52.2
444,1984.0,15-19 Years,80.4
445,1984.0,15-19 Years,80.4
446,1984.0,15-19 Years,80.4
447,1984.0,15-19 Years,80.4
448,1984.0,15-19 Years,80.4


- Drop the duplicated rows

In [335]:
mortality_to_clean = mortality_to_clean.drop_duplicates(keep='first')

- The duplicated rows are gone

In [336]:
mortality_to_clean[mortality_to_clean.duplicated(keep=False)]

,Year,AgeGroup,DeathRate


### Find Missing Values